In [ ]:
%pip install "requests[socks]"


In [ ]:
!python -c "import sys; print(sys.executable)"

In [ ]:
import requests
from pathlib import Path
import socket

In [ ]:
host = "127.0.0.1"
port = 9050

try:
    with socket.create_connection((host, port), timeout=5):
        print("proxy port is reachable")
except OSError as e:
    print(f"proxy connection failed: {e}")

proxies = {
    "http": "socks5h://127.0.0.1:9050",
    "https": "socks5h://127.0.0.1:9050",
}

In [ ]:
output_dir = Path("responses")
output_dir.mkdir(parents=True, exist_ok=True)

for request_number in range(2005,2026):
    print(request_number)
    url = f"https://web.archive.org/cdx/search/cdx?url=https://www.imdb.com/chart/top/&from={request_number}&to={request_number}&output=json&fl=timestamp,original,statuscode,mimetype,digest"

    try:
        response = requests.get(
            url,
            proxies=proxies,
            timeout=60,
        )
        response.raise_for_status()

        output_file = output_dir / f"response_{request_number}.txt"
        output_file.write_bytes(response.content)

        print(f"OK {request_number}: {output_file}")

    except requests.RequestException as e:
        print(f"ERROR :/ {request_number}: {e}")

In [ ]:
def getPage(url, folder, exact_date, page_name, group_name, file_extension):
    proxies = {
        "http": "socks5h://127.0.0.1:9050",
        "https": "socks5h://127.0.0.1:9050",
    }

    output_dir = Path(folder)
    output_dir.mkdir(parents=True, exist_ok=True)

    try:
        response = requests.get(
            url,
            proxies=proxies,
            timeout=60,
        )
        
        response.raise_for_status()

        output_file = output_dir / f"{exact_date}_{page_name}.{file_extension}"
        output_file.write_bytes(response.content)

        print(f"OK {exact_date}: {output_file}")

    except requests.RequestException as e:
        print(f"ERROR :/ {exact_date}: {e}")

In [ ]:
import pandas as pd
import json
from uuid import uuid4
from time import sleep
import random

df = pd.DataFrame(columns=["exact_date"])

SAVE_TO_FILE = False

for i in range(2005,2006):
    print(i)
    with open(f"responses/response_{i}.txt", 'r') as file:
        print("\n\n__________________",i,"__________________\n\n")
        parsed_file_as_array = json.load(file)
        for idx, content in enumerate(parsed_file_as_array):
            if(idx!=0):
                date = content[0]
                getPage(
                    f"https://web.archive.org/web/{date}/https://www.imdb.com/chart/top/", 
                    f"{i}_top_250", 
                    f"{date}", 
                    "top_250", 
                    f"{i}_top_250", 
                    "html"
                )
                sleep(random.uniform(1, 4))
                
                # if SAVE_TO_FILE:
                #     df.loc[len(df)] = {
                #         "exact_date": content[0]
                #     }

# if SAVE_TO_FILE:
#     df.to_csv(f"responses-{uuid4()}.csv", index=False)